# TRANSPAN Training School - Day 1

## Application of AI in Personalized Medicine for Pancreatic Cancer
### Practical session 1: building and judging a diagnostic model

**14 September 2026 - online**

---

### Read this before we start

**You do not need to know Python.** This session is built so that you can follow
it by reading, running and making small changes to code that is already written
for you. Every exercise gives you the exact lines to paste. Nothing is graded and
nothing is timed against you.

**How a notebook works.** A notebook is a list of boxes called *cells*. Grey
cells contain code. To run a cell, click inside it and press **Shift + Enter**.
The result appears underneath. Cells must be run **in order, from the top** -
each one depends on the ones before it.

**If something breaks.** Red text is an error message. Most of the time you have
either skipped a cell above, or made a typo. The fastest fix is: *Runtime ->
Restart and run all*, then continue from where you were. If that fails, say so in
the chat and keep reading - the fully worked notebook has every output already
saved, so you will not lose the thread.

**Marked in the notebook:**

- **Try it** boxes are yours. About two minutes each. The code you need is
  printed in the box - copy it, paste it where the cell says, run it.
- **Stop and think** boxes are for discussion. No code.

---

### The data

> Debernardi S, O'Brien H, Algahmdi AS, Malats N, Stewart GD, et al. (2020).
> A combination of urinary biomarker panel and PancRISK score for earlier
> detection of pancreatic cancer: a case-control study. *PLOS Medicine* 17(12):
> e1003489.

590 people in three groups: healthy controls, benign hepatobiliary disease, and
pancreatic ductal adenocarcinoma (PDAC). Four proteins measured in urine, plus
age, sex, and a blood test (CA19-9).

### Plan for the three hours

| | | approx. |
|---|---|---|
| 0 | Five minutes of Python, and loading the data | 25 min |
| 1 | Looking at the data before modelling it | 30 min |
| 2 | Which columns are we allowed to use? | 20 min |
| | *break* | 15 min |
| 3 | A first model | 30 min |
| 4 | Why a good AUC can still be a useless test | 30 min |
| 5 | Choosing a threshold, and calibration | 20 min |
| 6 | Explaining a prediction | 15 min |
| 7 | Work in pairs | 20 min |

---
## 0. Setup

Run the next cell. It loads the tools we need. You do not have to understand it.
It should print a few version numbers after about 20 seconds.

In [ ]:
try:
    import shap
except ImportError:
    !pip install -q shap
    import shap

import warnings
warnings.filterwarnings("ignore")

import numpy as np              # numbers
import pandas as pd             # tables
import matplotlib.pyplot as plt # plots

RANDOM_STATE = 42               # makes every result reproducible
np.random.seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (6.5, 4.2)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

print("Ready.")
print("pandas", pd.__version__, "| shap", shap.__version__)

### 0.1 Five minutes of Python

Four ideas cover almost everything you will see today.

**A variable is a name for a value.** `age = 62` means "from now on, `age` refers
to 62". The `=` is not equality, it is assignment.

**A function does something to a value.** You write its name, then round
brackets, then what you give it: `print(age)`, `round(3.14159, 2)`.

**A comment starts with `#`.** Python ignores the rest of that line. Comments are
notes for humans.

**A method is a function that belongs to a thing.** You reach it with a dot:
`table.mean()` means "the mean of that table". The empty brackets are still
needed.

Run the cell below and read it line by line.

In [ ]:
age = 62                       # a variable holding a number
name = "patient A"             # a variable holding text
markers = ["LYVE1", "REG1B", "TFF1"]   # a list of three items

print(name, "is", age, "years old")
print("We will use", len(markers), "markers:", markers)
print("The first one is", markers[0])   # Python counts from 0, not 1

> ### Try it 1 - 2 minutes
>
> Change `age` to your own guess of the average age of a pancreatic cancer patient, then add one line that prints the **last** marker in the list. In Python, `-1` means the last item.
>
> **The code to paste:**
>
> ```python
> print("The last one is", markers[-1])
> ```

In [ ]:
age = 62                       # <-- change this number
name = "patient A"
markers = ["LYVE1", "REG1B", "TFF1"]

print(name, "is", age, "years old")
print("We will use", len(markers), "markers:", markers)
print("The first one is", markers[0])

# YOUR TURN: paste the line from the hint box here

### 0.2 Loading the data

The next cell finds the dataset. It is long, and you can ignore its contents.
The only line that matters is the message it prints at the end, which tells you
whether you are working with the real data or with a simulated stand-in.

In [ ]:
from pathlib import Path

DATA_URL = None
CANDIDATES = ["Debernardi et al 2020 data.csv", "debernardi.csv",
              "urinary_biomarkers.csv"]


def make_stand_in(seed=20260914):
    """Simulated data with the schema of the Debernardi et al. (2020) release."""
    rng = np.random.default_rng(seed)
    dx = np.array([1] * 183 + [2] * 208 + [3] * 199)
    n = len(dx)
    ln = lambda mu, s: np.exp(rng.normal(mu, s))
    pick = lambda a, b, c: np.select([dx == 1, dx == 2], [a, b], default=c)

    age = np.clip(pick(rng.normal(52, 15, n), rng.normal(59, 14, n),
                       rng.normal(66, 11, n)), 26, 89).round().astype(int)
    sex = np.where(rng.random(n) < np.where(dx == 3, .55, .46), "M", "F")
    cohort = np.where(rng.random(n) < .52, "Cohort1", "Cohort2")

    LYVE1 = ln(pick(-.55, -.15, 1.15), .95) * (1 + .004 * (age - 59))
    REG1B = ln(pick(2.9, 3.4, 4.8), 1.15) * (1 + .004 * (age - 59))
    TFF1 = ln(pick(4.5, 5.1, 6.4), 1.30)
    creat = ln(np.where(dx == 3, -.55, -.40), .75)
    REG1A = np.where(cohort == "Cohort1", ln(pick(3.9, 4.4, 5.6), 1.30), np.nan)
    ca = ln(pick(2.2, 2.8, 5.6), 1.7)
    ca = np.where(rng.random(n) < pick(.18, .72, .93), ca, np.nan)

    df = pd.DataFrame({
        "sample_id": [f"S{i:03d}" for i in range(1, n + 1)],
        "patient_cohort": cohort,
        "sample_origin": rng.choice(["BPTB", "ESP", "LIV", "UCL"], n,
                                    p=[.55, .20, .15, .10]),
        "age": age, "sex": sex, "diagnosis": dx,
        "stage": np.where(dx == 3, rng.choice(
            ["I", "IA", "IB", "II", "IIA", "IIB", "III", "IV"], n,
            p=[.03, .03, .05, .10, .14, .30, .20, .15]), None),
        "benign_sample_diagnosis": np.where(dx == 2, rng.choice(
            ["Chronic pancreatitis", "Gallstones", "Pancreatic cyst",
             "Benign biliary stricture", "Acute pancreatitis", "IPMN"], n,
            p=[.34, .22, .16, .12, .09, .07]), None),
        "plasma_CA19_9": ca.round(2), "creatinine": creat.round(5),
        "LYVE1": LYVE1.round(5), "REG1B": REG1B.round(3),
        "REG1A": REG1A.round(3), "TFF1": TFF1.round(3),
    })
    return df.sample(frac=1.0, random_state=1).reset_index(drop=True)


found = next((p for p in CANDIDATES if Path(p).exists()), None)
if found:
    df = pd.read_csv(found)
    print(f"Loaded the real dataset from: {found}")
elif DATA_URL:
    df = pd.read_csv(DATA_URL)
    print(f"Loaded the real dataset from: {DATA_URL}")
else:
    df = make_stand_in()
    print("!! No real data file found - using the SIMULATED stand-in dataset.")
    print("!! Numbers below are for teaching only, not clinical results.")

print("\nThe table has", df.shape[0], "rows and", df.shape[1], "columns.")

`df` is now a **DataFrame**: a table, like a spreadsheet. Rows are patients,
columns are variables.

`df.head(8)` shows the first eight rows. It is the first thing to run on any new
dataset.

In [ ]:
df.head(8)

> ### Try it 2 - 2 minutes
>
> Show only the first **3** rows instead of 8. Then, on a new line, show the *last* 3 rows using `df.tail(3)`. Only the last line of a cell is displayed as a table, so wrap the first one in `print()` or run them one at a time.
>
> **The code to paste:**
>
> ```python
> df.tail(3)
> ```

In [ ]:
print(df.head(3))     # <-- change the 8 to a 3 in your own version

# YOUR TURN: paste the line from the hint box here

The columns you need to know:

| column | meaning |
|---|---|
| `age`, `sex` | demographics |
| `diagnosis` | **1** = healthy control, **2** = benign disease, **3** = PDAC |
| `creatinine`, `LYVE1`, `REG1B`, `TFF1` | the four urinary proteins, in ng/ml |
| `REG1A` | a fifth protein, measured only in part of the study |
| `plasma_CA19_9` | the standard blood test, measured only in some patients |
| `stage`, `benign_sample_diagnosis` | recorded **after** diagnosis - remember this |

---
## 1. Look at the data before you model it

The most common failure in clinical machine learning is not a bad algorithm. It
is a model built on data that nobody looked at first. Thirty minutes here saves a
retraction later.

### 1.1 Selecting one column

A single column is written `df["age"]`. Square brackets, and the name in quotes.
Once you have a column you can ask it for a summary: `.mean()`, `.median()`,
`.min()`, `.max()`, `.count()`.

In [ ]:
print("Number of patients :", df["age"].count())
print("Mean age           :", df["age"].mean().round(1))
print("Youngest            :", df["age"].min())

> ### Try it 3 - 2 minutes
>
> Add two lines: the **median** age, and the age of the **oldest** patient. Is the mean close to the median? If they differ a lot, the distribution is skewed.
>
> **The code to paste:**
>
> ```python
> print("Median age         :", df["age"].median())
> print("Oldest             :", df["age"].max())
> ```

In [ ]:
print("Number of patients :", df["age"].count())
print("Mean age           :", df["age"].mean().round(1))
print("Youngest            :", df["age"].min())

# YOUR TURN: paste the two lines from the hint box here

### 1.2 Counting the groups

`value_counts()` counts how many times each value appears in a column. Adding
`normalize=True` turns those counts into proportions.

In [ ]:
counts = df["diagnosis"].value_counts().sort_index()
print("Number of patients in each group:")
print(counts)

> ### Try it 4 - 2 minutes
>
> Add the proportions as well, so we can see the balance between the three groups. Then look hard at the proportion in group 3.
>
> **The code to paste:**
>
> ```python
> proportions = df["diagnosis"].value_counts(normalize=True).sort_index()
> print("\nAs proportions:")
> print(proportions.round(3))
> ```

In [ ]:
counts = df["diagnosis"].value_counts().sort_index()
print("Number of patients in each group:")
print(counts)

# YOUR TURN: paste the three lines from the hint box here

> ### Stop and think
>
> About one patient in three in this dataset has pancreatic cancer.
>
> **Write that number down.** It is a property of how the study was built - cases
> were recruited deliberately, one for one, against controls. It is not the
> frequency of pancreatic cancer in any population you would ever test.
>
> In section 4 this single number destroys a result that will look excellent.

### 1.3 Filtering rows

`df[df["diagnosis"] == 3]` keeps only the rows where the condition is true. Note
the **double** equals sign: one `=` assigns, two `==` compares.

In [ ]:
pdac = df[df["diagnosis"] == 3]
print("PDAC patients:", len(pdac))
print("Their mean age:", pdac["age"].mean().round(1))

> ### Try it 5 - 2 minutes
>
> Do the same for the healthy controls (`diagnosis == 1`) and compare the two mean ages. Cancer patients are older. Keep that in mind - a model can score well just by learning age.
>
> **The code to paste:**
>
> ```python
> controls = df[df["diagnosis"] == 1]
> print("Controls:", len(controls))
> print("Their mean age:", controls["age"].mean().round(1))
> ```

In [ ]:
pdac = df[df["diagnosis"] == 3]
print("PDAC patients:", len(pdac))
print("Their mean age:", pdac["age"].mean().round(1))

# YOUR TURN: paste the three lines from the hint box here

### 1.4 Missing values

`isna()` marks every empty cell, and `sum()` adds those marks up per column.

In [ ]:
missing = df.isna().sum()
print("Empty cells per column:")
print(missing[missing > 0])

Two of those are easy: `stage` is empty for everyone who does not have cancer,
and `benign_sample_diagnosis` is empty for everyone who does not have benign
disease. Those columns exist only *after* a diagnosis.

`plasma_CA19_9` is the interesting one. Let us see *who* it is missing for.
`groupby("diagnosis")` splits the table into the three groups and computes
something within each.

In [ ]:
by_group = df.groupby("diagnosis")["plasma_CA19_9"].apply(lambda s: s.isna().mean())
print("Fraction of CA19-9 values that are missing, per group:")
print(by_group.round(3))

> ### Try it 6 - 2 minutes
>
> Do the same check for `REG1A`. Its missingness has a different cause - it was only measured in one of the two recruitment cohorts. Compare the two patterns: one depends on the diagnosis, the other does not.
>
> **The code to paste:**
>
> ```python
> by_group_reg1a = df.groupby("diagnosis")["REG1A"].apply(lambda s: s.isna().mean())
> print("\nSame check for REG1A:")
> print(by_group_reg1a.round(3))
> ```

In [ ]:
by_group = df.groupby("diagnosis")["plasma_CA19_9"].apply(lambda s: s.isna().mean())
print("Fraction of CA19-9 values that are missing, per group:")
print(by_group.round(3))

# YOUR TURN: paste the three lines from the hint box here

> ### Stop and think - 3 minutes, use the chat
>
> CA19-9 is missing for most healthy controls and for almost none of the cancer
> patients. That is not an accident. A blood test gets ordered when someone is
> already worried about the patient.
>
> So the mere **presence** of a CA19-9 value tells you something about the
> outcome - information that would not exist at the moment you actually want to
> use the model. A model that picks up on it will look brilliant and be useless.
>
> **Question for you:** in your own field, what is the equivalent? Name one
> measurement whose *availability* depends on the answer you are trying to
> predict.

### 1.5 The shape of the biomarkers

`describe()` gives the standard summary of every numeric column at once.

In [ ]:
markers = ["creatinine", "LYVE1", "REG1B", "TFF1"]
print(df[markers].describe().round(2))

Look at the gap between the median (`50%`) and the maximum. These values span
three or four orders of magnitude, which is normal for protein concentrations
and very awkward for a model. A picture makes it obvious.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, m in zip(axes, markers):
    ax.hist(df[m].dropna(), bins=40, color="#4c72b0")
    ax.set_title(m)
    ax.set_yticks([])
fig.suptitle("Raw values: everything is crushed against the left edge", y=1.06)
plt.tight_layout()
plt.show()

Every marker is squashed into the far left with a long tail. The standard fix is
the **logarithm**, which turns "ten times bigger" into "one step further along".

> ### Try it 7 - 2 minutes
>
> Redraw the same four histograms on the log scale by wrapping the values in `np.log10(...)`. Compare the two pictures. Which one would you rather hand to a model?
>
> **The code to paste:**
>
> ```python
> ax.hist(np.log10(df[m].dropna()), bins=40, color="#55a868")
> ```

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, m in zip(axes, markers):
    # YOUR TURN: replace the line below with the one from the hint box
    ax.hist(df[m].dropna(), bins=40, color="#4c72b0")
    ax.set_title(m)
    ax.set_xlabel("log10 of concentration")
    ax.set_yticks([])
fig.suptitle("After a log transform: symmetric and well spread", y=1.06)
plt.tight_layout()
plt.show()

### 1.6 Is there any signal at all?

Before building anything, check whether the markers differ between the groups. If
they do not, no algorithm will save you.

In [ ]:
labels = {1: "control", 2: "benign", 3: "PDAC"}
colors = {1: "#55a868", 2: "#dd8452", 3: "#c44e52"}

fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
for ax, m in zip(axes, markers):
    groups = [np.log10(df.loc[df["diagnosis"] == d, m].dropna()) for d in (1, 2, 3)]
    box = ax.boxplot(groups, tick_labels=[labels[d] for d in (1, 2, 3)],
                     patch_artist=True, widths=0.6)
    for patch, d in zip(box["boxes"], (1, 2, 3)):
        patch.set_facecolor(colors[d])
        patch.set_alpha(0.75)
    ax.set_title(m)
    ax.set_ylabel("log10 concentration")
fig.suptitle("Each marker, by diagnosis", y=1.05)
plt.tight_layout()
plt.show()

Three of the four rise clearly in PDAC. **Creatinine barely moves**, and that is
expected: it is in the panel to measure how dilute the urine sample was, not to
detect disease. Keeping it still helps, because it lets the model correct the
others for dilution.

> ### Try it 8 - 2 minutes
>
> Add `age` to the same picture, to see how strongly age alone separates the groups. Change the list of columns being plotted and drop the log transform (age is already on a sensible scale).
>
> **The code to paste:**
>
> ```python
> groups = [df.loc[df["diagnosis"] == d, "age"] for d in (1, 2, 3)]
> plt.boxplot(groups, tick_labels=["control", "benign", "PDAC"])
> plt.ylabel("age (years)")
> plt.title("Age by diagnosis")
> plt.show()
> ```

In [ ]:
# YOUR TURN: paste the five lines from the hint box here

---
## 2. Which columns are we allowed to use?

Decide the clinical question first, then keep only the variables that would exist
**at the moment the question is asked**.

**Our question:** given a urine sample and basic demographics, does this person
likely have PDAC?

The model therefore runs *before* the diagnosis exists. So these columns are
forbidden, no matter how well they predict:

- `stage` - only exists once cancer is confirmed
- `benign_sample_diagnosis` - only exists once benign disease is confirmed
- `plasma_CA19_9` - available, but only for people already under suspicion (1.4)
- `sample_id`, `patient_cohort`, `sample_origin` - identifiers and study centres

Using a column like that is called **leakage**. Here is what it looks like.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier

y_all = (df["diagnosis"] == 3).astype(int)     # 1 if PDAC, else 0

# A single, deliberately silly feature: "was a stage recorded for this patient?"
silly = pd.DataFrame({"has_stage": df["stage"].notna().astype(int)})
score = cross_val_score(DecisionTreeClassifier(random_state=RANDOM_STATE),
                        silly, y_all, cv=5, scoring="roc_auc").mean()
print(f"AUC using only 'was a stage recorded?': {score:.3f}")

A perfect score, from a column that exists only because the patient already had
the diagnosis we are pretending to predict.

It is obvious here. In a hospital database with 400 columns it is not obvious at
all, and this is how published models quietly die.

**The test to apply to every single feature:** *when* was this recorded, relative
to the outcome? If the answer is "after it" or "because of it", drop it.

> ### Try it 9 - 2 minutes
>
> Run the same test on `benign_sample_diagnosis`. It marks benign patients rather than cancer patients, so think about what a high AUC would mean here before you look at the number.
>
> **The code to paste:**
>
> ```python
> silly2 = pd.DataFrame({"has_benign_dx": df["benign_sample_diagnosis"].notna().astype(int)})
> score2 = cross_val_score(DecisionTreeClassifier(random_state=RANDOM_STATE),
>                         silly2, y_all, cv=5, scoring="roc_auc").mean()
> print(f"AUC using only 'was a benign diagnosis recorded?': {score2:.3f}")
> ```

In [ ]:
# YOUR TURN: paste the four lines from the hint box here

### 2.1 Building the clean table

Two objects from here on:

- `X` - the **inputs**: one row per patient, one column per feature we are allowed
  to use.
- `y` - the **answer**: 1 if the patient has PDAC, 0 otherwise.

Every machine learning library in Python expects exactly this pair.

In [ ]:
FEATURES = ["age", "sex", "creatinine", "LYVE1", "REG1B", "TFF1"]
LOG_COLS = ["creatinine", "LYVE1", "REG1B", "TFF1"]

X = df[FEATURES].copy()
X["sex"] = (X["sex"] == "M").astype(int)   # models need numbers: M becomes 1
X[LOG_COLS] = np.log(X[LOG_COLS])          # the log transform from 1.5

y = (df["diagnosis"] == 3).astype(int)     # 1 = PDAC, 0 = not PDAC

print("X has", X.shape[0], "rows and", X.shape[1], "columns")
print("y has", int(y.sum()), "cancer patients out of", len(y))
X.head()

---
### Break - 15 minutes

Leave the screen. When we come back, we build the model.

---
## 3. A first model

Two rules matter more than any choice of algorithm.

**Rule 1: split the data before you look at it.** Keep part of the data locked
away. Every decision you make while looking at those patients - which features,
which threshold, which model - inflates the performance you will eventually
report.

**Rule 2: start with the boring model.** If a neural network cannot beat plain
logistic regression on 590 patients, the network was never the answer.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,          # keep 25% locked away
    stratify=y,              # same cancer proportion in both halves
    random_state=RANDOM_STATE)

print(f"training set: {len(X_train)} patients, {y_train.sum()} with cancer")
print(f"test set    : {len(X_test)} patients, {y_test.sum()} with cancer")

`stratify=y` is the part people forget. Without it, a random split can easily put
too few cancer patients in one half, and every number afterwards becomes noise.

### 3.1 Fitting logistic regression

A `Pipeline` chains steps together. Ours has two: put all features on a common
scale, then fit the model. Chaining matters because the scaling is then computed
from the training patients only, never from the locked-away ones.

`fit()` learns from data. `predict_proba()` returns a probability for each
patient.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])

logreg.fit(X_train, y_train)
print("Model fitted.")

### 3.2 How good is it, before we touch the test set?

**Cross-validation** splits the training data into five parts, trains on four,
tests on the fifth, and rotates. You get five scores instead of one, and the
spread between them tells you how much to trust the average.

The score we use is the **AUC**: the probability that a randomly chosen cancer
patient gets a higher score than a randomly chosen non-cancer patient. 0.5 is
a coin flip, 1.0 is perfect.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(logreg, X_train, y_train, cv=cv, scoring="roc_auc")

print("The five fold scores:", scores.round(3))
print(f"Average AUC: {scores.mean():.3f}")

> ### Try it 10 - 2 minutes
>
> Print the **standard deviation** of the five scores as well. Always report it. An average of 0.95 with a spread of 0.06 is a much weaker claim than the average alone suggests, and the spread is the honest half.
>
> **The code to paste:**
>
> ```python
> print(f"Spread across folds (std): {scores.std():.3f}")
> ```

In [ ]:
print("The five fold scores:", scores.round(3))
print(f"Average AUC: {scores.mean():.3f}")

# YOUR TURN: paste the line from the hint box here

### 3.3 What did it learn?

Because all the features were put on a common scale, the model's coefficients can
be compared directly. A positive value pushes the prediction toward cancer.

In [ ]:
coefficients = pd.Series(logreg.named_steps["model"].coef_[0], index=X.columns)
coefficients.sort_values().plot.barh(color="#4c72b0")
plt.axvline(0, color="black", lw=0.8)
plt.title("What the model relies on")
plt.xlabel("effect on the log-odds of cancer")
plt.tight_layout()
plt.show()

### 3.4 Is a fancier model better?

Gradient boosting is the usual reflex for table-shaped data. Let us check whether
the reflex is right here, with six features and about 440 training patients.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb = HistGradientBoostingClassifier(
    max_depth=3, learning_rate=0.06, max_iter=300,
    l2_regularization=1.0, random_state=RANDOM_STATE)

gb_scores = cross_val_score(gb, X_train, y_train, cv=cv, scoring="roc_auc")
gb.fit(X_train, y_train)

print(f"Logistic regression : {scores.mean():.3f}  (spread {scores.std():.3f})")
print(f"Gradient boosting   : {gb_scores.mean():.3f}  (spread {gb_scores.std():.3f})")

Whichever way the difference falls, it is smaller than the spread between folds.
With this much data you simply cannot tell the two apart, and saying one is
better would be reading noise.

That is a finding worth reporting rather than hiding. In clinical work the
simpler model wins on everything else that matters: it is easier to explain, to
validate elsewhere, and to get past a regulator.

> ### Try it 11 - 2 minutes
>
> Make the boosted model more complex by setting `max_depth=8` and re-run the comparison. More capacity, fewer patients per decision. Does the score go up or down, and what does that tell you about reaching for bigger models on small clinical datasets?
>
> **The code to paste:**
>
> ```python
> gb_deep = HistGradientBoostingClassifier(
>     max_depth=8, learning_rate=0.06, max_iter=300,
>     l2_regularization=1.0, random_state=RANDOM_STATE)
> deep_scores = cross_val_score(gb_deep, X_train, y_train, cv=cv, scoring='roc_auc')
> print(f'Deeper gradient boosting: {deep_scores.mean():.3f} (spread {deep_scores.std():.3f})')
> ```

In [ ]:
# YOUR TURN: paste the five lines from the hint box here

---
## 4. Why a good AUC can still be a useless test

Now, and only now, we open the test set.

In [ ]:
from sklearn.metrics import (roc_curve, roc_auc_score, precision_recall_curve,
                             average_precision_score, confusion_matrix)

p_lr = logreg.predict_proba(X_test)[:, 1]   # probability of cancer, per patient
p_gb = gb.predict_proba(X_test)[:, 1]

print(f"Logistic regression : AUC {roc_auc_score(y_test, p_lr):.3f}")
print(f"Gradient boosting   : AUC {roc_auc_score(y_test, p_gb):.3f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
for name, p, c in [("logistic regression", p_lr, "#4c72b0"),
                   ("gradient boosting", p_gb, "#c44e52")]:
    fpr, tpr, _ = roc_curve(y_test, p)
    ax1.plot(fpr, tpr, color=c, label=f"{name} ({roc_auc_score(y_test, p):.3f})")
    prec, rec, _ = precision_recall_curve(y_test, p)
    ax2.plot(rec, prec, color=c,
             label=f"{name} ({average_precision_score(y_test, p):.3f})")

ax1.plot([0, 1], [0, 1], "k--", lw=0.8)
ax1.set(xlabel="false alarm rate", ylabel="sensitivity", title="ROC curve")
ax2.axhline(y_test.mean(), color="k", ls="--", lw=0.8)
ax2.set(xlabel="sensitivity (recall)", ylabel="precision (PPV)",
        title="Precision-Recall curve")
ax1.legend(); ax2.legend()
plt.tight_layout()
plt.show()

Both curves look excellent. Now bring back the number from section 1.2.

**In this dataset, about 34% of people have cancer.** In a real high-risk group
you might screen - say, new-onset diabetes over 50 - the true frequency of PDAC
is around **1%**. In the general population it is far lower still.

Sensitivity and specificity do not change with frequency. **The chance that a
positive result is real does.** The next cell applies Bayes' rule to see how much.

In [ ]:
def ppv(sensitivity, specificity, prevalence):
    # Chance that a positive result is a real case
    true_positives = sensitivity * prevalence
    false_positives = (1 - specificity) * (1 - prevalence)
    return true_positives / (true_positives + false_positives)


# Set the model to catch 90% of cancers, as a screening test would be asked to
fpr, tpr, thresholds = roc_curve(y_test, p_lr)
i = np.argmax(tpr >= 0.90)
sens, spec, threshold = tpr[i], 1 - fpr[i], thresholds[i]
print(f"Operating point: sensitivity {sens:.2f}, specificity {spec:.2f}, "
      f"cut-off {threshold:.3f}\n")

rows = []
for prevalence in [0.34, 0.10, 0.05, 0.01, 0.001]:
    value = ppv(sens, spec, prevalence)
    rows.append({"how common cancer is": f"{prevalence:.1%}",
                 "chance a positive is real": f"{value:.1%}",
                 "false alarms per cancer found": f"{(1 - value) / value:.1f}"})
print(pd.DataFrame(rows).to_string(index=False))

> ### Stop and think - the most important table today
>
> **The model did not change. The AUC did not change. Only the population did.**
>
> The same test goes from clinically plausible in the study sample to sending
> dozens of healthy people for imaging per cancer found, once you apply it where
> cancer is actually rare.
>
> This is why a paper reporting only an AUC on a case-control sample tells you
> almost nothing about whether the test is usable. It is also why this particular
> panel is positioned for **stratifying patients who are already symptomatic or
> high-risk**, not for screening the general population.
>
> **Discuss:** at what "chance a positive is real" would you send a patient for a
> contrast CT? Would your answer change if the next step were a biopsy instead?

> ### Try it 12 - 2 minutes
>
> Add 2% and 0.5% to the list of frequencies, so you can see where the test stops being usable for you. Pick the value closest to the population you actually work with.
>
> **The code to paste:**
>
> ```python
> for prevalence in [0.34, 0.10, 0.05, 0.02, 0.01, 0.005, 0.001]:
> ```

In [ ]:
rows = []
# YOUR TURN: replace the line below with the one from the hint box
for prevalence in [0.34, 0.10, 0.05, 0.01, 0.001]:
    value = ppv(sens, spec, prevalence)
    rows.append({"how common cancer is": f"{prevalence:.1%}",
                 "chance a positive is real": f"{value:.1%}",
                 "false alarms per cancer found": f"{(1 - value) / value:.1f}"})
print(pd.DataFrame(rows).to_string(index=False))

---
## 5. Choosing a cut-off, and calibration

### 5.1 The cut-off is a clinical decision

The model gives a probability. Turning it into "yes" or "no" needs a cut-off, and
the usual default of 0.5 is an arbitrary convention, not a medical judgement.

In [ ]:
def report(name, probabilities, cut_off):
    predictions = (probabilities >= cut_off).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, predictions).ravel()
    print(f"--- {name} (cut-off {cut_off:.3f}) ---")
    print(f"  cancers found   {tp:>3}   cancers MISSED  {fn:>3}")
    print(f"  false alarms    {fp:>3}   correct all-clear {tn:>3}")
    print(f"  sensitivity {tp/(tp+fn):.2f}  specificity {tn/(tn+fp):.2f}"
          f"  PPV {tp/(tp+fp):.2f}\n")


report("default", p_lr, 0.50)
report("tuned for 90% sensitivity", p_lr, threshold)

Moving the cut-off does not make the model better. It slides you along a fixed
curve, buying caught cancers at the price of false alarms. The only real question
is which of the two mistakes you would rather make, and that is not a statistical
question.

> ### Try it 13 - 2 minutes
>
> Try a much stricter cut-off of 0.80, where the model only flags patients it is very confident about. How many cancers get missed? Would you accept that trade?
>
> **The code to paste:**
>
> ```python
> report("strict", p_lr, 0.80)
> ```

In [ ]:
report("default", p_lr, 0.50)
report("tuned for 90% sensitivity", p_lr, threshold)

# YOUR TURN: paste the line from the hint box here

### 5.2 Calibration

The AUC only asks whether cancer patients score *higher*. **Calibration** asks
something different and more useful: when the model says 30%, do 30 out of 100
such patients actually have cancer?

A model can rank patients perfectly and still be badly calibrated. A wrong
probability handed to a clinician is worse than no number at all.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

fig, ax = plt.subplots(figsize=(5.5, 5))
for name, p, c in [("logistic regression", p_lr, "#4c72b0"),
                   ("gradient boosting", p_gb, "#c44e52")]:
    observed, predicted = calibration_curve(y_test, p, n_bins=8,
                                            strategy="quantile")
    ax.plot(predicted, observed, "o-", color=c,
            label=f"{name} (Brier {brier_score_loss(y_test, p):.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="perfect")
ax.set(xlabel="probability the model predicted",
       ylabel="how often it actually happened",
       title="Calibration")
ax.legend()
plt.tight_layout()
plt.show()

Points on the dotted line mean the probabilities can be believed. Logistic
regression usually sits close to it. Boosted trees tend to push probabilities
toward 0 and 1 and often need a correction step afterwards.

With around 150 test patients each point rests on fewer than 20 people, so the
line wobbles. Do not over-read it, and always say how many patients are behind
each point.

---
## 6. Explaining a single prediction

"The model says 0.82" is not usable in a clinic. **SHAP** breaks each individual
prediction into per-feature contributions, which is the form the question
actually takes: why *this* patient?

The next cell takes about 15 seconds.

In [ ]:
background = shap.sample(X_train, 80, random_state=RANDOM_STATE)
explainer = shap.Explainer(gb.predict_proba, background)
shap_values = explainer(X_test)
cancer_side = shap_values[..., 1]

shap.plots.beeswarm(cancer_side, show=False)
plt.title("Every test patient, every feature")
plt.tight_layout()
plt.show()

Each dot is one patient. Its position shows how much that feature pushed that
patient's risk up (right) or down (left). The colour is the feature's value: red
is high, blue is low. Red dots far to the right mean "a high value of this marker
raises predicted risk".

Now one patient on their own.

In [ ]:
i = int(np.argmax(p_gb))     # the highest-risk patient in the test set
truth = "PDAC" if y_test.iloc[i] == 1 else "not PDAC"
print(f"Predicted probability of cancer: {p_gb[i]:.3f}   (truth: {truth})")

shap.plots.waterfall(cancer_side[i], show=False)
plt.tight_layout()
plt.show()

> ### Try it 14 - 2 minutes
>
> Look at the patient the model was **most wrong** about: the highest predicted risk among those who did not have cancer. Which feature misled it? These are the cases a clinician will ask you about.
>
> **The code to paste:**
>
> ```python
> wrong = np.argmax(np.where(y_test.values == 0, p_gb, -1))
> print(f'Predicted {p_gb[wrong]:.3f} but this patient did NOT have cancer')
> shap.plots.waterfall(cancer_side[wrong], show=False)
> plt.tight_layout()
> plt.show()
> ```

In [ ]:
# YOUR TURN: paste the five lines from the hint box here

> ### Stop and think
>
> SHAP explains **the model**, not the disease. A large contribution means the
> model leaned on that feature. It does not mean the protein causes cancer, and
> it does not mean that changing the protein would change the outcome.
>
> Presenting SHAP rankings as biological discoveries is one of the most common
> serious errors in the clinical AI literature. Now you will notice it.

---
## 7. Work in pairs - 20 minutes

Pick **one**. Everything you need has appeared already; copy the relevant cells
and change them. Post your headline number in the shared document with one
sentence on what surprised you.

**A. The harder, more realistic question.** Drop the healthy controls and
separate PDAC from benign disease only. In clinic nobody sends healthy people for
a urine panel - the real question is whether *this* unwell patient has cancer.
Does the AUC fall? Enough to change what you would do with the model?

**B. Does the blood test help?** Keep only patients who actually have a CA19-9
value, add `np.log(df["plasma_CA19_9"])` as a seventh feature, and refit. Then
argue about whether that comparison is fair, given section 1.4.

**C. Age alone.** Build a model with `age` as the only feature. How much of the
performance was the biomarkers, and how much was just knowing who is older?

Start from the worked example for A below.

In [ ]:
# --- Your workspace ---------------------------------------------------------
# For exercise A, the two lines that set up the harder problem are:
#
#   keep = df["diagnosis"].isin([2, 3])
#   X_hard = X.loc[keep]
#   y_hard = (df.loc[keep, "diagnosis"] == 3).astype(int)
#
# Then reuse train_test_split, the Pipeline, and roc_auc_score exactly as in
# sections 3 and 4.

---
## What to take away

In order of importance, and none of it is about algorithms:

1. **Look at the data first.** Missing values are a variable in their own right,
   and they usually know the answer.
2. **Ask when each column was recorded.** Anything written down after the
   diagnosis, or because of it, is leakage - however good it makes your numbers.
3. **AUC says almost nothing about clinical usefulness.** How common the disease
   is decides whether a positive result means anything.
4. **The cut-off is a clinical decision**, not a software default.
5. **Calibration is a separate property from ranking**, and it is the one that
   matters when a probability is shown to a human.
6. **A simple model you can explain** beats a slightly sharper one you cannot,
   everywhere a decision follows from the output.

Tomorrow, 15 September: many more features than patients. Gene expression from
TCGA-PAAD, survival modelling, and what happens to all of today's good practice
when the data gets wide.

*Questions afterwards: pmmartins@iscac.pt*